# InduSense — Sprint 1 / TP1 : cartes d'identité des fichiers sources

**Objectif** : pour chaque fichier déposé dans `datas/`, produire une carte d'identité Markdown dans
`artifacts/ingestion/identities/<AAAAMMJJ>/<nom_fichier>_<HHMMSS>.md` contenant :
source, taille, horaires de traitement, encodage, nombre de lignes et de colonnes,
liste des colonnes avec leur type inféré, et statistiques (min, max, moyenne, médiane) des colonnes numériques et dates.

Le code est factorisé en fonctions courtes, chacune avec une seule responsabilité.

## 0. Vue d'ensemble du flux

```
datas/*.csv
   │
   ├─ 2. get_source()             → Source : chemin absolu
   ├─ 3. human_size()             → Filesize : "9999.99 Ko | Mo | Go"
   │     detect_charset()         → Charset : "UTF-8" | "Windows-1252" | "ISO-8859-1" | ...
   ├─ 4. read_csv_raw()           → DataFrame brut (séparateur détecté) → Rows / Cols Number
   ├─ 5. infer_type()             → Column list (type inféré de chaque colonne)
   ├─ 6. column_stats()           → min / max (+ moyenne / médiane pour les numériques)
   ├─ 7. render_identity()        → texte Markdown de la carte (ProcessBeginAt / ProcessEndAt)
   │     write_identity()         → artifacts/ingestion/identities/AAAAMMJJ/<fichier>_HHMMSS.md
   └─ 8. build_csv_identity()     → enchaîne tout pour un fichier
```

## 1. Imports et paramètres

| Bibliothèque   | Origine         | Usage                                                    |
|----------------|-----------------|----------------------------------------------------------|
| `pathlib.Path` | stdlib          | Chemins, taille, création des dossiers                  |
| `datetime`     | stdlib          | Horaires de traitement, nom des dossiers/fichiers       |
| `csv`          | stdlib          | `csv.Sniffer` pour détecter le séparateur (`,` `;` `\t`) |
| `pandas`       | `uv add pandas` | Lecture, types, statistiques                             |
| `re`           | stdlib          | expressions régulières : repérer les colonnes identifiants (_id), les booléens, les formats de date |

In [34]:
import csv
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

DATA_DIR = Path("datas")
OUT_ROOT = Path("artifacts/ingestion/identities")

# Paramètres de l'inférence de type (section 5)
NA_TOKENS = {"", "na", "n/a", "nan", "null", "none", "-"}  # valeurs considérées comme manquantes
PARSE_RATE_MIN = 0.95          # part minimale de valeurs convertibles pour retenir un type
CATEGORY_MAX_UNIQUE = 30       # au-delà, un texte n'est plus une catégorie
MIN_ROWS_FOR_IDENTITY = 20     # échantillon minimal pour conclure "valeurs toutes uniques = identifiant"

In [35]:
print(f"pandas {pd.__version__}")
print(f"Dossier des données  : {DATA_DIR.resolve()}  (existe : {DATA_DIR.exists()})")
print(f"Dossier de sortie    : {OUT_ROOT.resolve()}")
print(f"Fichiers à traiter   : {[p.name for p in sorted(DATA_DIR.glob('*.csv'))]}")
print(f"Valeurs manquantes   : {sorted(NA_TOKENS)}")
print(f"Seuils               : PARSE_RATE_MIN={PARSE_RATE_MIN}, CATEGORY_MAX_UNIQUE={CATEGORY_MAX_UNIQUE}, "
      f"MIN_ROWS_FOR_IDENTITY={MIN_ROWS_FOR_IDENTITY}")

pandas 3.1.1
Dossier des données  : C:\dev\00-Fomation_IA\TP1\indusense\datas  (existe : True)
Dossier de sortie    : C:\dev\00-Fomation_IA\TP1\indusense\artifacts\ingestion\identities
Fichiers à traiter   : ['releves_incidents.csv', 'telemetry.csv']
Valeurs manquantes   : ['', '-', 'n/a', 'na', 'nan', 'none', 'null']
Seuils               : PARSE_RATE_MIN=0.95, CATEGORY_MAX_UNIQUE=30, MIN_ROWS_FOR_IDENTITY=20


## 2. Source

La source d'un fichier est son **chemin absolu** : `Path.resolve()` transforme le chemin relatif
(`datas/telemetry.csv`) en chemin complet, indépendant du dossier courant.

In [36]:
def get_source(file_path: Path) -> str:
    """Retourne le chemin absolu du fichier source."""
    return str(file_path.resolve())

In [37]:
for file_path in sorted(DATA_DIR.glob("*.csv")):
    print(f"Source: {get_source(file_path)}")

Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\releves_incidents.csv
Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\telemetry.csv


## 3. Métadonnées du fichier : taille et encodage

**Taille** : exprimée en unités binaires (1 Ko = 1024 octets), en Ko, Mo ou Go selon la taille du fichier.

**Encodage** : on teste d'abord UTF-8 (avec ou sans BOM). Si le décodage échoue, le fichier est
dans un encodage 8 bits : s'il contient des octets entre `0x80` et `0x9F`, c'est du Windows-1252
(ces octets sont des caractères imprimables en Windows-1252, des caractères de contrôle en ISO-8859-1),
sinon ISO-8859-1 (Latin-1).

In [38]:
def human_size(n_bytes: int) -> str:
    """Taille lisible en unités binaires (1 Ko = 1024 octets)."""
    size = n_bytes / 1024
    for unit in ("Ko", "Mo", "Go"):
        if size < 1024 or unit == "Go":
            return f"{size:.2f} {unit}"
        size /= 1024


def detect_charset(path: Path) -> str:
    """Détecte l'encodage d'un fichier texte sans dépendance externe."""
    raw = path.read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        return "UTF-8 (BOM)"
    try:
        raw.decode("utf-8")
        return "ASCII (compatible UTF-8)" if raw.isascii() else "UTF-8"
    except UnicodeDecodeError:
        if any(0x80 <= b <= 0x9F for b in raw):
            return "Windows-1252"
        return "ISO-8859-1"


# Correspondance libellé affiché -> nom d'encodage Python
PY_ENCODING = {"UTF-8 (BOM)": "utf-8-sig", "UTF-8": "utf-8", "ASCII (compatible UTF-8)": "utf-8",
               "Windows-1252": "cp1252", "ISO-8859-1": "latin-1"}

In [39]:
for file_path in sorted(DATA_DIR.glob("*.csv")):
    print(f"{file_path.name:25} Filesize: {human_size(file_path.stat().st_size):>10}"
          f"   Charset: {detect_charset(file_path)}")

releves_incidents.csv     Filesize:  134.64 Ko   Charset: UTF-8
telemetry.csv             Filesize:    7.94 Mo   Charset: ASCII (compatible UTF-8)


## 4. Lecture du CSV

Le séparateur est détecté automatiquement (`,` `;` tabulation ou `|`).
Le fichier est lu **en texte** (`dtype=str`) : on veut observer la donnée brute,
sans que pandas convertisse silencieusement les décimales à virgule ou les dates.

In [40]:
def detect_separator(path: Path, encoding: str) -> str:
    with path.open(encoding=encoding, newline="") as f:
        sample = f.read(50_000)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error:
        return ","


def read_csv_raw(path: Path, encoding: str) -> pd.DataFrame:
    sep = detect_separator(path, encoding)
    return pd.read_csv(path, sep=sep, encoding=encoding, dtype=str,
                       keep_default_na=False, na_filter=False)

In [41]:
# Les DataFrames bruts sont gardés en mémoire pour les tests des sections suivantes
raw_tables = {}
for file_path in sorted(DATA_DIR.glob("*.csv")):
    encoding = PY_ENCODING[detect_charset(file_path)]
    sep = detect_separator(file_path, encoding)
    df = read_csv_raw(file_path, encoding)
    raw_tables[file_path.name] = df
    print(f"{file_path.name:25} sep={sep!r}  encoding={encoding:8}  {df.shape[0]} lignes x {df.shape[1]} colonnes")

for name, df in raw_tables.items():
    print(f"\n=== {name} : 3 premières lignes ===")
    print(df.head(3).to_string())

releves_incidents.csv     sep=','  encoding=utf-8     1245 lignes x 18 colonnes
telemetry.csv             sep=','  encoding=utf-8     135626 lignes x 7 colonnes

=== releves_incidents.csv : 3 premières lignes ===
  incident_id        date   time  operator_name machine_id severity operator_badge                        comment       shift type_surchauffe type_baisse_pression type_vibration type_bruit_mecanique type_surconsommation type_blocage_mecanique type_alarme_capteur type_arret_urgence type_defaut_qualite
0  INC-000001  2025-06-01  05:42  Lucas Bernard    MACH-06        4         OP1002               chauffe anormale        nuit               1                    0              0                    0                    0                      0                   0                  0                   0
1  INC-000002  2025-06-01  21:08    Hugo Thomas    MACH-15        3         OP1004  micro-fuite / baisse pression  apres-midi               0                    1              0      

## 5. Inférence des types

Types retenus, testés dans cet ordre :

| Type | Règle |
|---|---|
| `EMPTY` | aucune valeur renseignée |
| `BOOLEAN` | au plus 2 valeurs, toutes dans {0/1, true/false, oui/non, yes/no} |
| `IDENTITY` | nom de colonne en `id`, `*_id`, `id_*`, `code`, `*_code` ou `matricule` |
| `INTEGER` / `FLOAT` | ≥ 95 % des valeurs convertibles en nombre (la virgule décimale est acceptée) |
| `TIME` | ≥ 95 % des valeurs au format heure seule `HH:MM` ou `HH:MM:SS` |
| `DATETIME` | ≥ 95 % des valeurs convertibles en date (ISO 8601, sinon format jour/mois) |
| `IDENTITY` | texte à valeurs toutes uniques (au moins 20 valeurs) |
| `CATEGORY` | texte avec peu de valeurs distinctes (≤ 30) |
| `TEXT` | autre texte |

La fonction renvoie aussi la série convertie, réutilisée pour les statistiques.

In [42]:
ID_NAME = re.compile(r"(^id$|_id$|^id_|^code$|_code$|^matricule$)", re.IGNORECASE)
TIME_PATTERN = re.compile(r"^\d{1,2}:\d{2}(:\d{2})?$")
BOOL_VALUES = {"0", "1", "true", "false", "oui", "non", "yes", "no"}


def present_values(s: pd.Series) -> pd.Series:
    """Valeurs renseignées (hors cellules vides / NaN et jetons de valeur manquante)."""
    s = s.dropna().astype(str).str.strip()
    return s[~s.str.lower().isin(NA_TOKENS)]


def to_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.str.replace(" ", "", regex=False)
                          .str.replace(",", ".", regex=False), errors="coerce")


def to_datetime(s: pd.Series) -> pd.Series:
    iso = pd.to_datetime(s, errors="coerce", format="ISO8601")
    if iso.notna().mean() >= PARSE_RATE_MIN:
        return iso
    return pd.to_datetime(s, errors="coerce", dayfirst=True, format="mixed")


def to_time(s: pd.Series) -> pd.Series:
    """Heures seules (HH:MM ou HH:MM:SS) en durée depuis minuit, sans date ajoutée."""
    s = s.where(s.str.count(":") == 2, s + ":00")
    return pd.to_timedelta(s, errors="coerce")


def infer_type(name: str, s: pd.Series) -> tuple[str, pd.Series | None]:
    """Renvoie (type inféré, série convertie ou None)."""
    values = present_values(s)
    if values.empty:
        return "EMPTY", None
    if values.str.lower().isin(BOOL_VALUES).all() and values.nunique() <= 2:
        return "BOOLEAN", None

    num = to_numeric(values)
    numeric = num.notna().mean() >= PARSE_RATE_MIN
    if ID_NAME.search(name):
        return "IDENTITY", (num if numeric else None)
    if numeric:
        is_int = (num.dropna() % 1 == 0).all()
        return ("INTEGER" if is_int else "FLOAT"), num

    # Heure seule : testée avant les dates, sinon pandas lui colle la date du jour
    if values.str.match(TIME_PATTERN).mean() >= PARSE_RATE_MIN:
        return "TIME", to_time(values)

    # Les dates contiennent forcément un séparateur ; on évite ainsi de tester du texte libre
    if values.str.contains(r"[-/:]").mean() >= PARSE_RATE_MIN:
        dates = to_datetime(values)
        if dates.notna().mean() >= PARSE_RATE_MIN:
            return "DATETIME", dates

    # Toutes valeurs uniques = identifiant, seulement sur un échantillon suffisant
    if values.is_unique and len(values) >= MIN_ROWS_FOR_IDENTITY:
        return "IDENTITY", None
    if values.nunique() <= CATEGORY_MAX_UNIQUE:
        return "CATEGORY", None
    return "TEXT", None

In [43]:
for name, df in raw_tables.items():
    print(f"\n=== {name} ===")
    print(f"{'Colonne':25} {'Type inféré':12} Exemple de valeur")
    for col in df.columns:
        col_type, _ = infer_type(col, df[col])
        print(f"{col:25} {col_type:12} {df[col].iloc[0]!r}")


=== releves_incidents.csv ===
Colonne                   Type inféré  Exemple de valeur
incident_id               IDENTITY     'INC-000001'
date                      DATETIME     '2025-06-01'
time                      TIME         '05:42'
operator_name             CATEGORY     'Lucas Bernard'
machine_id                IDENTITY     'MACH-06'
severity                  INTEGER      '4'
operator_badge            CATEGORY     'OP1002'
comment                   TEXT         'chauffe anormale'
shift                     CATEGORY     'nuit'
type_surchauffe           BOOLEAN      '1'
type_baisse_pression      BOOLEAN      '0'
type_vibration            BOOLEAN      '0'
type_bruit_mecanique      BOOLEAN      '0'
type_surconsommation      BOOLEAN      '0'
type_blocage_mecanique    BOOLEAN      '0'
type_alarme_capteur       BOOLEAN      '0'
type_arret_urgence        BOOLEAN      '0'
type_defaut_qualite       BOOLEAN      '0'

=== telemetry.csv ===
Colonne                   Type inféré  Exemple de va

## 6. Statistiques des colonnes numériques et dates

Pour chaque colonne `INTEGER`, `FLOAT`, `DATETIME` ou `TIME` : min et max.
Moyenne et médiane uniquement pour les numériques, conformément à la consigne (« — » sinon).
Les identifiants numériques (ex. `id_releve`) sont exclus : leur moyenne n'a pas de sens.

In [44]:
def format_time(t: pd.Timedelta) -> str:
    total = int(t.total_seconds())
    return f"{total // 3600:02d}:{total % 3600 // 60:02d}:{total % 60:02d}"


def column_stats(name: str, col_type: str, converted: pd.Series | None) -> dict | None:
    if converted is None or col_type not in {"INTEGER", "FLOAT", "DATETIME", "TIME"}:
        return None
    s = converted.dropna()
    if s.empty:
        return None
    # Dates et heures : min / max seulement (moyenne et médiane réservées aux numériques)
    if col_type == "DATETIME":
        fmt = lambda d: d.strftime("%Y-%m-%d %H:%M:%S")
        return {"col": name, "min": fmt(s.min()), "max": fmt(s.max()), "mean": "—", "median": "—"}
    if col_type == "TIME":
        return {"col": name, "min": format_time(s.min()), "max": format_time(s.max()),
                "mean": "—", "median": "—"}
    # Min / max gardent le type de la colonne (entier sans décimales)
    fmt_bound = (lambda x: f"{int(x)}") if col_type == "INTEGER" else (lambda x: f"{x:.2f}")
    return {"col": name, "min": fmt_bound(s.min()), "max": fmt_bound(s.max()),
            "mean": f"{s.mean():.2f}", "median": f"{s.median():.2f}"}

In [45]:
for name, df in raw_tables.items():
    print(f"\n=== {name} ===")
    print(f"{'Colonne':25} {'Type':9} {'Min':>20} {'Max':>20} {'Moyenne':>10} {'Médiane':>10}")
    for col in df.columns:
        col_type, converted = infer_type(col, df[col])
        st = column_stats(col, col_type, converted)
        if st:
            print(f"{col:25} {col_type:9} {st['min']:>20} {st['max']:>20} {st['mean']:>10} {st['median']:>10}")


=== releves_incidents.csv ===
Colonne                   Type                       Min                  Max    Moyenne    Médiane
date                      DATETIME   2025-06-01 00:00:00  2026-06-08 00:00:00          —          —
time                      TIME                  00:06:00             23:55:00          —          —
severity                  INTEGER                      2                    5       2.93       3.00

=== telemetry.csv ===
Colonne                   Type                       Min                  Max    Moyenne    Médiane
timestamp                 DATETIME   2025-06-01 00:00:00  2026-06-08 23:00:00          —          —
temperature_c             FLOAT                    32.00                80.00      48.18      48.05
pressure_bar              FLOAT                   159.98               215.81     199.77     199.87
voltage_mean_v            FLOAT                   220.90               242.00     227.63     227.42
rotation_mean_rpm         FLOAT               

## 7. Construction et écriture de la carte d'identité

`ProcessBeginAt` est noté avant la détection d'encodage, `ProcessEndAt` après la lecture,
l'inférence des types et les statistiques : il couvre donc tout le traitement du fichier.

In [46]:
def profile_dataframe(df: pd.DataFrame) -> tuple[list[dict], list[dict]]:
    """Types inférés et statistiques de chaque colonne d'un DataFrame brut."""
    columns, stats = [], []
    for name in df.columns:
        col_type, converted = infer_type(name, df[name])
        columns.append({"name": name, "type": col_type})
        st = column_stats(name, col_type, converted)
        if st:
            stats.append(st)
    return columns, stats


def render_header(path: Path, charset: str, begin: datetime, end: datetime) -> list[str]:
    """Étapes 1 à 4, communes à tous les fichiers (CSV et SQL)."""
    return [f"# Carte d'identité — {path.name}", "",
            f"- Source: {get_source(path)}",
            f"- Filesize : {human_size(path.stat().st_size)}",
            f"- ProcessBeginAt: {begin:%H:%M:%S}",
            f"- ProcessEndAt: {end:%H:%M:%S}",
            f"- Charset: {charset}"]


def render_identity(path: Path, charset: str, begin: datetime, end: datetime,
                    df: pd.DataFrame, columns: list[dict], stats: list[dict]) -> str:
    """Carte d'un fichier CSV au format Markdown."""
    lines = render_header(path, charset, begin, end)
    lines += [f"- Rows Number : {len(df)}",
              f"- Cols Number : {df.shape[1]}",
              "", "### Column list", "",
              "| Colname | Infered Type |", "|---|---|"]
    lines += [f"| {c['name']} | {c['type']} |" for c in columns]
    lines += ["", "### Numeric and date columns", ""]
    if stats:
        lines += ["| Col name | Min value | Max value | Mean value | Median value |",
                  "|---|---|---|---|---|"]
        lines += [f"| {s['col']} | {s['min']} | {s['max']} | {s['mean']} | {s['median']} |"
                  for s in stats]
    else:
        lines.append("_Aucune colonne numérique ou date._")
    return "\n".join(lines) + "\n"


def file_base(path: Path) -> str:
    """Nom du fichier sans AUCUNE extension : machine.sql.txt -> machine."""
    return path.name.split(".")[0]


def identity_path(path: Path, begin: datetime) -> Path:
    """artifacts/ingestion/identities/AAAAMMJJ/<fichier>_HHMMSS.md"""
    return OUT_ROOT / f"{begin:%Y%m%d}" / f"{file_base(path)}_{begin:%H%M%S}.md"


def write_identity(path: Path, content: str, begin: datetime) -> Path:
    out = identity_path(path, begin)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(content, encoding="utf-8")
    return out


def build_csv_identity(path: Path) -> Path:
    begin = datetime.now()
    charset = detect_charset(path)
    df = read_csv_raw(path, PY_ENCODING[charset])
    columns, stats = profile_dataframe(df)
    end = datetime.now()  # après lecture + inférence + statistiques
    content = render_identity(path, charset, begin, end, df, columns, stats)
    return write_identity(path, content, begin)

In [47]:
# Aperçu de la carte SANS l'écrire sur le disque (l'écriture se fait en section 8)
test_path = DATA_DIR / "releves_incidents.csv"
begin = datetime.now()
charset = detect_charset(test_path)
test_df = read_csv_raw(test_path, PY_ENCODING[charset])
columns, stats = profile_dataframe(test_df)
end = datetime.now()
print(f"Durée du traitement : {(end - begin).total_seconds():.2f} s")
print(f"Fichier qui sera écrit : {identity_path(test_path, begin)}\n")
print(render_identity(test_path, charset, begin, end, test_df, columns, stats))

Durée du traitement : 0.11 s
Fichier qui sera écrit : artifacts\ingestion\identities\20261120\releves_incidents_160108.md

# Carte d'identité — releves_incidents.csv

- Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\releves_incidents.csv
- Filesize : 134.64 Ko
- ProcessBeginAt: 16:01:08
- ProcessEndAt: 16:01:08
- Charset: UTF-8
- Rows Number : 1245
- Cols Number : 18

### Column list

| Colname | Infered Type |
|---|---|
| incident_id | IDENTITY |
| date | DATETIME |
| time | TIME |
| operator_name | CATEGORY |
| machine_id | IDENTITY |
| severity | INTEGER |
| operator_badge | CATEGORY |
| comment | TEXT |
| shift | CATEGORY |
| type_surchauffe | BOOLEAN |
| type_baisse_pression | BOOLEAN |
| type_vibration | BOOLEAN |
| type_bruit_mecanique | BOOLEAN |
| type_surconsommation | BOOLEAN |
| type_blocage_mecanique | BOOLEAN |
| type_alarme_capteur | BOOLEAN |
| type_arret_urgence | BOOLEAN |
| type_defaut_qualite | BOOLEAN |

### Numeric and date columns

| Col name | Min value | Max

## 8. Traitement des CSV du dossier `datas/`

In [48]:
csv_files = sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() in {".csv", ".tsv"})
print(f"{len(csv_files)} fichier(s) à traiter : {[p.name for p in csv_files]}\n")
for p in csv_files:
    out = build_csv_identity(p)
    print(f"{p.name:30} -> {out}  ({out.stat().st_size} octets)")

2 fichier(s) à traiter : ['releves_incidents.csv', 'telemetry.csv']

releves_incidents.csv          -> artifacts\ingestion\identities\20261120\releves_incidents_160114.md  (1122 octets)
telemetry.csv                  -> artifacts\ingestion\identities\20261120\telemetry_160114.md  (985 octets)


Aperçu d'une carte produite :

In [49]:
from IPython.display import Markdown, display
generated = sorted(OUT_ROOT.rglob("*.md"))
display(Markdown(generated[-1].read_text(encoding="utf-8")))

# Carte d'identité — telemetry.csv

- Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\telemetry.csv
- Filesize : 7.94 Mo
- ProcessBeginAt: 16:01:14
- ProcessEndAt: 16:01:17
- Charset: ASCII (compatible UTF-8)
- Rows Number : 135626
- Cols Number : 7

### Column list

| Colname | Infered Type |
|---|---|
| machine_id | IDENTITY |
| timestamp | DATETIME |
| temperature_c | FLOAT |
| pressure_bar | FLOAT |
| voltage_mean_v | FLOAT |
| rotation_mean_rpm | FLOAT |
| pieces_produced | INTEGER |

### Numeric and date columns

| Col name | Min value | Max value | Mean value | Median value |
|---|---|---|---|---|
| timestamp | 2025-06-01 00:00:00 | 2026-06-08 23:00:00 | — | — |
| temperature_c | 32.00 | 80.00 | 48.18 | 48.05 |
| pressure_bar | 159.98 | 215.81 | 199.77 | 199.87 |
| voltage_mean_v | 220.90 | 242.00 | 227.63 | 227.42 |
| rotation_mean_rpm | 1100.00 | 1900.00 | 1589.20 | 1590.37 |
| pieces_produced | 0 | 114 | 49.53 | 49.00 |


## 9. Pour aller plus loin : le script SQL

Bibliothèque retenue : **`sqlglot`**, en pur Python (aucune extension compilée), donc installable
quelle que soit la version de Python : `uv add sqlglot`.
Contrairement à `sqlparse`, qui ne fait que découper le texte en jetons, `sqlglot` construit un
arbre syntaxique : on y lit directement chaque `CREATE TABLE`, ses colonnes et leur type déclaré.

Démarche :
- **Étapes 1 à 4 réutilisées telles quelles** : `get_source`, `human_size`, horaires de traitement,
  `detect_charset` (en-tête produit par `render_header`, le même que pour les CSV).
- **Étape 5 remplacée** : au lieu de lire un tableau et d'inférer des types, on extrait les
  `CREATE TABLE` : tables, colonnes et types déclarés. Les `INSERT` et les `CREATE INDEX` sont ignorés.
- **Markdown adapté** : nombre de tables, puis pour chaque table son nombre de colonnes et le tableau
  `Colname | Declared Type`.

Remarque : `sqlglot` normalise l'écriture des types (`INTEGER` → `INT`, `NUMERIC(6,2)` → `DECIMAL(6, 2)`).

In [50]:
import sqlglot
from sqlglot import exp


def extract_create_tables(path: Path, encoding: str, dialect: str = "postgres") -> dict[str, dict[str, str]]:
    """Étape 5 (SQL) : {table: {colonne: type déclaré}} à partir des CREATE TABLE."""
    tables = {}
    for stmt in sqlglot.parse(path.read_text(encoding=encoding), read=dialect):
        if isinstance(stmt, exp.Create) and stmt.kind == "TABLE":
            schema = stmt.this                       # nom de la table + définitions de colonnes
            tables[schema.this.name] = {
                col.name: col.args["kind"].sql(dialect=dialect) if col.args.get("kind") else ""
                for col in schema.expressions if isinstance(col, exp.ColumnDef)
            }
    return tables


def render_sql_identity(path: Path, charset: str, begin: datetime, end: datetime,
                        tables: dict[str, dict[str, str]]) -> str:
    """Carte d'un script SQL : en-tête commun (étapes 1 à 4) + une section par table."""
    lines = render_header(path, charset, begin, end)
    lines.append(f"- Tables Number : {len(tables)}")
    for table, columns in tables.items():
        lines += ["", f"## Table `{table}`", "",
                  f"- Cols Number : {len(columns)}", "",
                  "| Colname | Declared Type |", "|---|---|"]
        lines += [f"| {col} | {col_type} |" for col, col_type in columns.items()]
    return "\n".join(lines) + "\n"


def build_sql_identity(path: Path) -> Path:
    begin = datetime.now()
    charset = detect_charset(path)
    tables = extract_create_tables(path, PY_ENCODING[charset])
    end = datetime.now()
    content = render_sql_identity(path, charset, begin, end, tables)
    return write_identity(path, content, begin)

In [51]:
# Test de l'étape 5 (SQL) : ce que sqlglot extrait des CREATE TABLE
sql_files = sorted(DATA_DIR.glob("*.sql*"))  # .sql ou .sql.txt
print(f"{len(sql_files)} script(s) SQL trouvé(s) : {[p.name for p in sql_files]}")
for p in sql_files:
    tables = extract_create_tables(p, PY_ENCODING[detect_charset(p)])
    print(f"\n=== {p.name} : {len(tables)} table(s) ===")
    for table, columns in tables.items():
        print(f"\nTable {table} ({len(columns)} colonnes)")
        for col, col_type in columns.items():
            print(f"   {col:30} {col_type}")

1 script(s) SQL trouvé(s) : ['machine.sql.txt']

=== machine.sql.txt : 2 table(s) ===

Table machine (11 colonnes)
   machine_code                   VARCHAR(16)
   commissioning_date             DATE
   max_daily_capacity             INT
   max_hourly_capacity_pieces     INT
   model                          VARCHAR(32)
   production_line                VARCHAR(16)
   location                       VARCHAR(16)
   criticality                    VARCHAR(8)
   is_active                      BOOLEAN
   created_at                     TIMESTAMPTZ
   updated_at                     TIMESTAMPTZ

Table maintenance (11 colonnes)
   maintenance_id                 INT
   machine_code                   VARCHAR(16)
   maintenance_at                 TIMESTAMPTZ
   maintenance_type               VARCHAR(16)
   action_type                    VARCHAR(32)
   component                      VARCHAR(64)
   description                    TEXT
   related_incident_id            VARCHAR(16)
   duration_hours    

In [52]:
# Écriture de la carte de chaque script SQL, puis affichage de son contenu
for p in sql_files:
    out = build_sql_identity(p)
    print(f"{p.name:30} -> {out}  ({out.stat().st_size} octets)\n")
    print(out.read_text(encoding="utf-8"))

machine.sql.txt                -> artifacts\ingestion\identities\20261120\machine_160149.md  (1085 octets)

# Carte d'identité — machine.sql.txt

- Source: C:\dev\00-Fomation_IA\TP1\indusense\datas\machine.sql.txt
- Filesize : 22.41 Ko
- ProcessBeginAt: 16:01:49
- ProcessEndAt: 16:01:50
- Charset: UTF-8
- Tables Number : 2

## Table `machine`

- Cols Number : 11

| Colname | Declared Type |
|---|---|
| machine_code | VARCHAR(16) |
| commissioning_date | DATE |
| max_daily_capacity | INT |
| max_hourly_capacity_pieces | INT |
| model | VARCHAR(32) |
| production_line | VARCHAR(16) |
| location | VARCHAR(16) |
| criticality | VARCHAR(8) |
| is_active | BOOLEAN |
| created_at | TIMESTAMPTZ |
| updated_at | TIMESTAMPTZ |

## Table `maintenance`

- Cols Number : 11

| Colname | Declared Type |
|---|---|
| maintenance_id | INT |
| machine_code | VARCHAR(16) |
| maintenance_at | TIMESTAMPTZ |
| maintenance_type | VARCHAR(16) |
| action_type | VARCHAR(32) |
| component | VARCHAR(64) |
| descri